In [1]:
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

d:\coding\ai_learning\krishnaik\ultimate_rag_bootcamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Step 1: Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [3]:
### Step 2: Vector Store
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

### Step 3: MMR Retriever
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={
    "k": 5,
})
retriever


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14522.92it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026B092B6BA0>, search_type='mmr', search_kwargs={'k': 5})

In [4]:
### Step 4: LLM and Prompt
import os
from dotenv import load_dotenv

load_dotenv()

# llm = init_chat_model(model="openai:gpt-4o-mini")
llm = init_chat_model(model="granite4:latest", model_provider="ollama")
llm

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, model='granite4:latest')

In [5]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, model='granite4:latest')
| StrOutputParser()

In [6]:
query_expansion_chain.invoke({"query": "Langchain memory"})

'Here is an expanded version of the query that includes synonyms, related concepts, and additional context to help with more comprehensive document retrieval:\n\nQuery: Langchain memory, persistent state, contextual information storage, dynamic knowledge base, state management system\n\nContext: In natural language processing (NLP) applications built using the Langchain framework, a "memory" component is essential for maintaining relevant information across multiple interactions or steps within a larger AI workflow. Memory allows the model to retain context, track progress towards goals, store temporary variables, and persist state between calls.\n\nKey aspects of Langchain\'s memory include:\n\n1. Persistent State: The ability to maintain a record of important data that survives across different API calls, sessions, or runs of an application using Langchain components like LLM Chains, Agents, or Interfaces.\n\n2. Contextual Information Storage: Temporarily storing information relevant

In [7]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm=llm, prompt=answer_prompt)

In [8]:
## Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap(
        {
            "input": lambda x: x["input"],
            "context": lambda x: retriever.invoke(
                query_expansion_chain.invoke(
                    {"query": x["input"]}
                )
            )
        }
    )
    | document_chain
)


In [9]:
# Step 6: Run query
query = {"input": "What types of memory does Langchain support?"}
print(query_expansion_chain.invoke({"query": query}))
response = rag_pipeline.invoke(query)
print("Answer:\n", response)

**Expanded Query**

```json
{
  "input": [
    "What types of memory are supported by Langchain?",
    "Which in‑memory data structures can be integrated with Langchain?",
    "What kinds of external memory stores can Langchain connect to?",
    "Can you list the different forms of state that Langchain's Memory class can hold?",
    "How does Langchain handle persistent or long‑term storage as part of its memory system?"
  ]
}
```

**Explanation**

- **Memory Types**: The query is broad, so we add variations that focus on *types* (e.g., “kinds,” “forms”) and *supports* (e.g., “can hold”).  
- **Specificity**: By asking for *in‑memory*, *external*, or *persistent* memory, the response will cover Langchain’s modular design where different backends can be plugged in.  
- **Contextual Clues**: Terms like “state,” “data structures,” and “storage” help the model retrieve pages that discuss how Langchain abstracts various memory components (e.g., Python dictionaries, vector stores, file syste

In [10]:
query = {"input": "What types of memory does CrewAI support?"}
print(query_expansion_chain.invoke({"query": query}))
response = rag_pipeline.invoke(query)
print("Answer:\n", response)

{
  "input": [
    {
      "query": "CrewAI",
      "context": "A cutting-edge artificial intelligence platform that provides advanced machine learning models and tools for developers and businesses."
    },
    {
      "query": "types of memory supported",
      "context": "Refers to the different types of data storage or processing capabilities provided by CrewAI's AI models, enabling them to handle various kinds of input and context during interactions."
    }
  ]
}
Answer:
 The provided context does not mention anything about the types of memory that **CrewAI** supports. It focuses on features such as multi‑step workflow handling, support for multiple LLM backends, streaming, parallel execution, and asynchronous tool invocation, but it does not discuss any specifics related to memory management or storage capabilities within CrewAI. 

If you have additional details about the memory aspects (e.g., whether it supports in‑memory caches, persistent storage, GPU/TPU memory, etc.), those

In [11]:
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query": query}))
response = rag_pipeline.invoke(query)
print("Answer:\n", response)

**Expanded Query**

```json
{
  "input": [
    "CrewAI",
    "Crew AI system",
    "Crew Artificial Intelligence platform",
    "Crew AI deployment",
    "Crew AI integration",
    "Crew AI workflow management",
    "Crew AI automation capabilities",
    "Crew AI agent orchestration",
    "Crew AI task delegation",
    "Crew AI virtual assistant functionality",
    "Crew AI chatbot implementation",
    "Crew AI natural language processing",
    "Crew AI conversational interface",
    "Crew AI intelligent automation",
    "Crew AI workflow automation",
    "Crew AI integration architecture",
    "Crew AI API endpoints",
    "Crew AI RESTful services",
    "Crew AI microservices",
    "Crew AI distributed computing model",
    "Crew AI agent lifecycle management",
    "Crew AI error handling mechanisms",
    "Crew AI performance optimization techniques"
  ]
}
```

**Explanation**

- **Primary Term**: The original query centered on the term **“CrewAI agents.”**  
  - This is expanded to i